In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from geopy.geocoders import Nominatim
import time
from sklearn.preprocessing import LabelEncoder

## 도로 데이터 로드

In [ ]:
df = pd.read_csv("road_data.csv")
df

In [ ]:
df.columns

In [ ]:
df.drop(columns=["DSTA_LGLD_INFO", "DSTA_ADSTRD_INFO"], inplace=True)
df

In [ ]:
df['DTCT_PD'] = pd.to_datetime(df['DTCT_PD'])

# 날짜 분리
df['YEAR'] = df['DTCT_PD'].dt.year
df['MONTH'] = df['DTCT_PD'].dt.month
df['DAY'] = df['DTCT_PD'].dt.day
df['HOUR'] = df['DTCT_PD'].dt.hour

# 요일 (0=월요일, 6=일요일)
df['WEEKDAY'] = df['DTCT_PD'].dt.weekday

# 계절
def get_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'autumn'

df['SEASON'] = df['MONTH'].apply(get_season)

# 출퇴근 시간대 여부
def commute_flag(hour):
    if 8 <= hour <= 9:        
        return 1
    elif 18 <= hour <= 19:   
        return 1
    else:
        return 0

df['COMMUTE'] = df['HOUR'].apply(commute_flag)

In [ ]:
df

In [ ]:
df.rename(columns={"DTCT_PD" : "DATE", "PLC_LTTD" : "LAT", "PLC_LGTD" : "LON"}, inplace=True)
df

In [ ]:
# 최근 7/30/90일 Rolling Count (포트홀 기준)
pothole_df = df[df['damage_type'] == 'pothole'].copy()

# 날짜 오름차순 정렬
pothole_df = pothole_df.sort_values(['ROAD_INFO', 'DTCT_PD'])

pothole_df['COUNT_7D'] = pothole_df.groupby('ROAD_INFO')['DTCT_PD'] \
    .rolling('7D').count().reset_index(0, drop=True)

pothole_df['COUNT_30D'] = pothole_df.groupby('ROAD_INFO')['DTCT_PD'] \
    .rolling('30D').count().reset_index(0, drop=True)

pothole_df['COUNT_90D'] = pothole_df.groupby('ROAD_INFO')['DTCT_PD'] \
    .rolling('90D').count().reset_index(0, drop=True)

# 다시 원본 df로 merge
df = df.merge(
    pothole_df[['DTCT_UID','COUNT_7D','COUNT_30D','COUNT_90D']],
    on='DTCT_UID',
    how='left'
)